# Data Augmentation Analysis — CINIC-10 CNN

Visual comparison of standard and advanced augmentation techniques.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join('..', 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from utils import get_device, set_seeds

set_seeds(42)
device = get_device()

DATA_DIR  = os.path.join('..', 'data')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'valid')
EPOCHS = 3  # increase for final submission
BATCH_SIZE = 32
print(f'TRAIN_DIR: {TRAIN_DIR}')
print(f'Using device: {device}')
print('Setup complete')

## 1. Visual: Augmentation Effect on Sample Images

In [ ]:
from augmentation_studies import apply_cutout_augmentation, apply_cutmix_augmentation

# Load two sample images from the first class
sample_class = sorted(os.listdir(TRAIN_DIR))[0]
sample_files = sorted(os.listdir(os.path.join(TRAIN_DIR, sample_class)))

def load_img(path):
    return np.array(Image.open(path).convert('RGB').resize((32, 32))).astype(np.float32) / 255.0

orig1 = load_img(os.path.join(TRAIN_DIR, sample_class, sample_files[0]))
orig2 = load_img(os.path.join(TRAIN_DIR, sample_class, sample_files[1]))

cutout = apply_cutout_augmentation(orig1.copy(), mask_size=8)
cutmix = apply_cutmix_augmentation(orig1.copy(), orig2.copy())

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, img, title in zip(axes, [orig1, cutout, cutmix], ['Original', 'Cutout', 'CutMix']):
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(title)
    ax.axis('off')
plt.suptitle(f'Augmentation effects on: {sample_class}')
plt.tight_layout()
plt.show()

## 2. Augmentation Performance Comparison

In [ ]:
from model_architecture import create_baseline_cnn
from augmentation_studies import compare_augmentation_approaches

results = compare_augmentation_approaches(
    create_baseline_cnn, TRAIN_DIR, VAL_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE
)
print('Standard results:')
print(pd.DataFrame(results['standard'])[['augmentation', 'val_accuracy']])
print('\nAdvanced results:')
print(pd.DataFrame(results['advanced'])[['augmentation', 'val_accuracy']])

In [ ]:
from augmentation_studies import visualize_augmentation_results

all_results = results['standard'] + results['advanced']
visualize_augmentation_results(pd.DataFrame(all_results))

## Summary

- **No augmentation (minimal)**: baseline; typically overfits fastest
- **Standard (flip + crop + color)**: good generalization improvement at low cost
- **Cutout**: regularizes by forcing the model to use all spatial regions